# V2

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

def load_and_prepare_data(diagnosis_path, labs_path, clinical_path, demographics_path, drugs_path=None, is_control=False):
    """Load and prepare all required datasets. If drugs_path is provided, load the Drugs dataset as well."""
    diagnosis_df = pd.read_parquet(diagnosis_path)
    labs_df = pd.read_parquet(labs_path)
    clinical_df = pd.read_parquet(clinical_path)
    demographics_df = pd.read_parquet(demographics_path)
    
    # Add label column (1 for case, 0 for control)
    label = 0 if is_control else 1
    for df in [diagnosis_df, labs_df, clinical_df]:
        df['label'] = label

    drugs_df = None
    if drugs_path is not None:
        drugs_df = pd.read_parquet(drugs_path)
        drugs_df['label'] = label
    
    return diagnosis_df, labs_df, clinical_df, demographics_df, drugs_df

def check_organomegaly(diagnosis_df, person_id):
    """Check for presence of organomegaly conditions"""
    organomegaly_codes = {
        'Organomegaly': ['R19.00', 'R19.05', 'R19.06', 'R19.09'],
        'Hepatosplenomegaly': ['R16.2'],
        'Hepatomegaly': ['R16.0'],
        'Splenomegaly': ['R16.1']
    }
    
    person_diagnoses = diagnosis_df[diagnosis_df['PERSONID'] == person_id]
    results = {}
    
    for condition, codes in organomegaly_codes.items():
        has_condition = person_diagnoses['ICDCODE'].isin(codes).any()
        results[condition] = int(has_condition)
    
    return results, any(results.values())

def get_cbc_parameters(labs_df, drugs_df, person_id):
    """Get CBC parameters and check their values.
    
    If the patient has received velaglucerase alfa or imiglucerase parenteral (as per the Drugs dataset),
    only labs obtained before the first medication order date are considered.
    Otherwise, all available labs are used.
    """
    labs_params = {
        'WBC': (4.5, 11),
        'Hgb': (132, 173),
        'Platelet': (140, 400),
        'Neutro Auto #': (1.8, 7.7)
    }
    
    # Check if patient has taken one of the enzyme replacement medications
    use_pre_med_labs = False
    med_date = None
    if drugs_df is not None:
        person_drugs = drugs_df[drugs_df['PERSONID'] == person_id]
        # Check for medications (case insensitive)
        med_mask = person_drugs['ORDERMNEMONIC'].str.contains(
            'velaglucerase alfa|imiglucerase parenteral', 
            case=False, na=False
        )
        if med_mask.any():
            # Get the earliest medication order date
            med_dates = pd.to_datetime(person_drugs[med_mask]['ORDERDATE'], errors='coerce')
            med_date = med_dates.min()
            use_pre_med_labs = True

    # Filter labs: if medication was given, only use labs before that date.
    if use_pre_med_labs and med_date is not None:
        # It is assumed that labs_df has an 'ORDERDATE' column; adjust if needed.
        person_labs = labs_df[
            (labs_df['PERSONID'] == person_id) &
            (pd.to_datetime(labs_df['ORDERDATE'], errors='coerce') < med_date)
        ].copy()
    else:
        person_labs = labs_df[labs_df['PERSONID'] == person_id].copy()
    
    results = {}
    abnormal_flags = {}

    for param, (lower, upper) in labs_params.items():
        # If enzyme therapy exists, attempt to get pre-med labs for this parameter.
        if use_pre_med_labs and med_date is not None:
            param_labs = labs_df[
                (labs_df['PERSONID'] == person_id) &
                (pd.to_datetime(labs_df['ORDERDATE'], errors='coerce') < med_date) &
                (labs_df['TASKASSAY'].str.contains(param, na=False))
            ].copy()
            # If no pre-med labs exist for this parameter, then use all available labs.
            if param_labs.empty:
                param_labs = labs_df[
                    (labs_df['PERSONID'] == person_id) &
                    (labs_df['TASKASSAY'].str.contains(param, na=False))
                ].copy()
        else:
            param_labs = labs_df[
                (labs_df['PERSONID'] == person_id) &
                (labs_df['TASKASSAY'].str.contains(param, na=False))
            ].copy()

        if not param_labs.empty:
            param_labs['RESULTVALUE_NUM'] = pd.to_numeric(param_labs['RESULTVALUE'], errors='coerce')
            param_labs['ORDERDATE'] = pd.to_datetime(param_labs['ORDERDATE'], errors='coerce')
            param_labs = param_labs.sort_values('ORDERDATE')

            # First, try to select a lab value that is below the reference range.
            below = param_labs[param_labs['RESULTVALUE_NUM'] < lower]
            if not below.empty:
                selected = below.iloc[-1]
            else:
                # If none are below, check for values above the reference range.
                above = param_labs[param_labs['RESULTVALUE_NUM'] > upper]
                if not above.empty:
                    selected = above.iloc[-1]
                else:
                    # If no abnormal lab is found, default to the most recent lab value.
                    selected = param_labs.iloc[-1]

            results[param] = selected['RESULTVALUE_NUM']
            abnormal_flags[param] = (selected['RESULTVALUE_NUM'] < lower or selected['RESULTVALUE_NUM'] > upper)
        else:
            results[param] = None
            abnormal_flags[param] = False

    has_abnormal = any(abnormal_flags.values())
    return results, has_abnormal


def check_growth_parameters(clinical_df, person_id):
    """Check growth parameters using only Height/Weight Centile"""
    person_clinical = clinical_df[clinical_df['PERSONID'] == person_id]
    
    centile_entries = person_clinical[
        person_clinical['TASKASSAY'] == 'Height/Weight Centile'
    ].copy()
    
    if not centile_entries.empty:
        centile_values = pd.to_numeric(centile_entries['EVENTRESULT'], errors='coerce')
        has_low_centile = (centile_values < 3).any()
    else:
        has_low_centile = False
    
    return has_low_centile

def apply_expert_rules(diagnosis_df, labs_df, clinical_df, drugs_df, person_id):
    """Apply the expert rules and return detailed results"""
    person_diagnoses = diagnosis_df[diagnosis_df['PERSONID'] == person_id]
    
    # Check organomegaly
    organomegaly_results, has_organomegaly = check_organomegaly(diagnosis_df, person_id)
    # Get CBC parameters (using pre-medication labs if applicable)
    cbc_results, has_low_cbc = get_cbc_parameters(labs_df, drugs_df, person_id)
    
    # Growth related diagnoses (based on ICD codes)
    growth_retardation = person_diagnoses['ICDCODE'].isin(
        ['R62.50', 'R62.51', 'R62.52', 'R62.59']
    ).any()
    failure_to_thrive = person_diagnoses['ICDCODE'].isin(['R62.7']).any()
    
    # Other conditions
    lung_disease = person_diagnoses['ICDCODE'].isin(
        ['J84.9', 'J84.10', 'J84.112', 'J84.114', 'J84.848']
    ).any()
    hypotonia = person_diagnoses['ICDCODE'].isin(['P94.1', 'P94.2']).any()
    developmental_delay = person_diagnoses['ICDCODE'].isin(['R62.50', 'R62.59']).any()
    
    # Apply expert rules:
    rule1 = has_organomegaly and has_low_cbc
    rule2 = growth_retardation and rule1
    rule3 = (growth_retardation or failure_to_thrive or check_growth_parameters(clinical_df, person_id)) and rule1 and rule2
    rule4 = lung_disease and rule1
    rule5 = rule1 and (hypotonia or developmental_delay)
    
    return {
        'rule1': int(rule1),
        'rule2': int(rule2),
        'rule3': int(rule3),
        'rule4': int(rule4),
        'rule5': int(rule5),
        **organomegaly_results,
        **cbc_results,
        'growth_retardation': int(growth_retardation),
        'failure_to_thrive': int(failure_to_thrive),
        'interstitial_lung_disease': int(lung_disease),
        'hypotonia': int(hypotonia),
        'developmental_delay': int(developmental_delay)
    }

def process_rare_disease_data(case_paths, control_paths):
    """Process the data and create final results DataFrame.
    
    Each of case_paths and control_paths is expected to be a 5-tuple:
        (diagnosis_path, labs_path, clinical_path, demographics_path, drugs_path)
    """
    # Unpack paths for cases and controls
    case_diag, case_labs, case_clin, case_demo, case_drugs = case_paths
    control_diag, control_labs, control_clin, control_demo, control_drugs = control_paths
    
    # Load data (include drugs dataset)
    case_data = load_and_prepare_data(case_diag, case_labs, case_clin, case_demo, drugs_path=case_drugs)
    control_data = load_and_prepare_data(control_diag, control_labs, control_clin, control_demo, drugs_path=control_drugs, is_control=True)
    
    results = []
    for data_set in [case_data, control_data]:
        diagnosis_df, labs_df, clinical_df, demographics_df, drugs_df = data_set
        unique_persons = diagnosis_df['PERSONID'].unique()
        
        for person_id in unique_persons:
            # Get all EPI numbers for this person
            epi_numbers = demographics_df[demographics_df['PERSONID'] == person_id]['EPI'].unique()
            epi_str = '; '.join(epi_numbers)
            
            # Get expert rules results
            rules_results = apply_expert_rules(diagnosis_df, labs_df, clinical_df, drugs_df, person_id)
            is_positive = any([rules_results[f'rule{i}'] for i in range(1, 6)])
            is_case = diagnosis_df[diagnosis_df['PERSONID'] == person_id]['label'].iloc[0] == 1
            
            confusion_category = 'TP' if is_case and is_positive else \
                                 'FN' if is_case and not is_positive else \
                                 'FP' if not is_case and is_positive else 'TN'
            
            results.append({
                'PERSONID': person_id,
                'EPI': epi_str,
                'LABEL': int(is_case),
                'PREDICTED': int(is_positive),
                'CONFUSION_CATEGORY': confusion_category,
                'RULE1': rules_results['rule1'],
                'RULE2': rules_results['rule2'],
                'RULE3': rules_results['rule3'],
                'RULE4': rules_results['rule4'],
                'RULE5': rules_results['rule5'],
                'ORGANOMEGALY': rules_results['Organomegaly'],
                'HEPATOSPLENOMEGALY': rules_results['Hepatosplenomegaly'],
                'HEPATOMEGALY': rules_results['Hepatomegaly'],
                'SPLENOMEGALY': rules_results['Splenomegaly'],
                'WBC': rules_results['WBC'],
                'Hgb': rules_results['Hgb'],
                'PLATELET': rules_results['Platelet'],
                'NEUTRO_AUTO': rules_results['Neutro Auto #'],
                'GROWTH_RETARDATION': rules_results['growth_retardation'],
                'FAILURE_TO_THRIVE': rules_results['failure_to_thrive'],
                'INTERSTITIAL_LUNG_DISEASE': rules_results['interstitial_lung_disease'],
                'HYPOTONIA': rules_results['hypotonia'],
                'DEVELOPMENTAL_DELAY': rules_results['developmental_delay']
            })
    
    return pd.DataFrame(results)


rare_diseases = ['ASMD', 'Gaucher', 'Saposin']

for rd in rare_diseases: 
    case_paths = (
        f'../Datasets/{rd}/Diagnosis_optimized.parquet',
        f'../Datasets/{rd}/Labs_optimized.parquet',
        f'../Datasets/{rd}/Clinical_optimized.parquet',
        f'../Datasets/{rd}/Demographic_optimized.parquet',
        f'../Datasets/{rd}/Drugs_optimized.parquet'
    )
    control_paths = (
        '../Datasets/Gaucher-ASMD-Control/Diagnosis_optimized.parquet',
        '../Datasets/Gaucher-ASMD-Control/Labs_optimized.parquet',
        '../Datasets/Gaucher-ASMD-Control/Clinical_optimized.parquet',
        '../Datasets/Gaucher-ASMD-Control/Demographic_optimized.parquet',
        '../Datasets/Gaucher-ASMD-Control/Drugs_optimized.parquet'
    )

    # Process data
    results_df = process_rare_disease_data(case_paths, control_paths)

    results_df.to_excel(f'{rd}-results/{rd}_results_v3.xlsx', index=False)

In [11]:
RD = ['Saposin', 'Gaucher', 'ASMD']

for rd in RD:
    results_df = pd.read_excel(f'{rd}-results/{rd}_results_v3.xlsx')
    print(f"\nConfusion Matrix for {rd}")
    print(results_df['CONFUSION_CATEGORY'].value_counts())
    TP = results_df['CONFUSION_CATEGORY'].value_counts().get('TP', 0)
    FP = results_df['CONFUSION_CATEGORY'].value_counts().get('FP', 0)
    TN = results_df['CONFUSION_CATEGORY'].value_counts().get('TN', 0)

    if rd == 'Saposin':
        FN = 0
    else:
        FN = results_df['CONFUSION_CATEGORY'].value_counts().get('FN', 0)

    accuracy = (TP + TN) / (TP + TN + FP + FN)
    sensitivity = TP / (TP + FN)
    specificity = TN / (TN + FP)

    print(f"\nMetrics for {rd}:")
    print(f'Accuracy: {accuracy:.2f}')
    print(f'Sensitivity: {sensitivity:.2f}')
    print(f'Specificity: {specificity:.2f}')
    print("=====================================")


Confusion Matrix for Saposin
CONFUSION_CATEGORY
TN    521
FP     20
TP      3
Name: count, dtype: int64

Metrics for Saposin:
Accuracy: 0.96
Sensitivity: 1.00
Specificity: 0.96

Confusion Matrix for Gaucher
CONFUSION_CATEGORY
TN    521
FP     20
TP     10
FN      3
Name: count, dtype: int64

Metrics for Gaucher:
Accuracy: 0.96
Sensitivity: 0.77
Specificity: 0.96

Confusion Matrix for ASMD
CONFUSION_CATEGORY
TN    521
FP     20
TP      7
FN      1
Name: count, dtype: int64

Metrics for ASMD:
Accuracy: 0.96
Sensitivity: 0.88
Specificity: 0.96


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report

RD = ['Saposin', 'Gaucher', 'ASMD']

# Define your feature columns; adjust as needed.
features = [
    'ORGANOMEGALY', 'HEPATOSPLENOMEGALY', 'HEPATOMEGALY', 'SPLENOMEGALY',
    'WBC', 'Hgb', 'PLATELET', 'NEUTRO_AUTO',
    'GROWTH_RETARDATION', 'FAILURE_TO_THRIVE',
    'INTERSTITIAL_LUNG_DISEASE', 'HYPOTONIA', 'DEVELOPMENTAL_DELAY'
]

for rd in RD:
    results_df = pd.read_excel(f'{rd}-results/{rd}_results_v3.xlsx')


    X = results_df[features]
    y = results_df['LABEL']

    # Fill missing values as needed (using mean imputation here)
    X = X.fillna(X.mean())

    # Split data into training and testing sets.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Train a classifier.
    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_train, y_train)

    # Get predictions and compare.
    y_pred = clf.predict(X_test)
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))


[[108   0]
 [  1   0]]
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       108
           1       0.00      0.00      0.00         1

    accuracy                           0.99       109
   macro avg       0.50      0.50      0.50       109
weighted avg       0.98      0.99      0.99       109



c:\Users\Jalal\miniconda3\envs\cmhs\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Jalal\miniconda3\envs\cmhs\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Jalal\miniconda3\envs\cmhs\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[[107   1]
 [  1   2]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       108
           1       0.67      0.67      0.67         3

    accuracy                           0.98       111
   macro avg       0.83      0.83      0.83       111
weighted avg       0.98      0.98      0.98       111

[[108   0]
 [  2   0]]
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       108
           1       0.00      0.00      0.00         2

    accuracy                           0.98       110
   macro avg       0.49      0.50      0.50       110
weighted avg       0.96      0.98      0.97       110



c:\Users\Jalal\miniconda3\envs\cmhs\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Jalal\miniconda3\envs\cmhs\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Jalal\miniconda3\envs\cmhs\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
from pyod.models.hbos import HBOS
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

RD = ['Saposin', 'Gaucher', 'ASMD']

# Define your feature columns; adjust as needed.
features = [
    'ORGANOMEGALY', 'HEPATOSPLENOMEGALY', 'HEPATOMEGALY', 'SPLENOMEGALY',
    'WBC', 'Hgb', 'PLATELET', 'NEUTRO_AUTO',
    'GROWTH_RETARDATION', 'FAILURE_TO_THRIVE',
    'INTERSTITIAL_LUNG_DISEASE', 'HYPOTONIA', 'DEVELOPMENTAL_DELAY'
]

for rd in RD:
    results_df = pd.read_excel(f'{rd}-results/{rd}_results_v3.xlsx')

    X = results_df[features].copy()
    y = results_df['LABEL']

    # Fill missing values (using mean imputation here)
    X = X.fillna(X.mean())

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # Initialize and fit the HBOS model on the training data
    hbos = HBOS(contamination=0.1)
    hbos.fit(X_train)

    # HBOS outputs labels: 1 for outliers, 0 for inliers.
    # Predict on test data.
    y_pred = hbos.predict(X_test)

    # Evaluate performance
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))


[[149  13]
 [  0   2]]
              precision    recall  f1-score   support

           0       1.00      0.92      0.96       162
           1       0.13      1.00      0.24         2

    accuracy                           0.92       164
   macro avg       0.57      0.96      0.60       164
weighted avg       0.99      0.92      0.95       164

[[151  10]
 [  2   4]]
              precision    recall  f1-score   support

           0       0.99      0.94      0.96       161
           1       0.29      0.67      0.40         6

    accuracy                           0.93       167
   macro avg       0.64      0.80      0.68       167
weighted avg       0.96      0.93      0.94       167

[[144  18]
 [  0   3]]
              precision    recall  f1-score   support

           0       1.00      0.89      0.94       162
           1       0.14      1.00      0.25         3

    accuracy                           0.89       165
   macro avg       0.57      0.94      0.60       165
weigh

In [49]:
from pyod.models.copod import COPOD
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# List of disease result file prefixes (each file contains positives & controls)
RD = ['Saposin', 'Gaucher', 'ASMD']

# Define your feature columns.
features = [
    'ORGANOMEGALY', 'HEPATOSPLENOMEGALY', 'HEPATOMEGALY', 'SPLENOMEGALY',
    'WBC', 'Hgb', 'PLATELET', 'NEUTRO_AUTO',
    'GROWTH_RETARDATION', 'FAILURE_TO_THRIVE',
    'INTERSTITIAL_LUNG_DISEASE', 'HYPOTONIA', 'DEVELOPMENTAL_DELAY'
]

# Read and combine the Excel files.
dfs = []
for rd in RD:
    df = pd.read_excel(f'{rd}-results/{rd}_results_v3.xlsx')
    df['Disease'] = rd  # Optional: record disease type.
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

# Optionally, drop duplicate control patients if needed:
# combined_df = combined_df.drop_duplicates(subset='PERSONID')

# Separate features and label.
X = combined_df[features].copy()
y = combined_df['LABEL']

# Fill missing values.
X = X.fillna(X.mean())

# Split data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Initialize and fit the HBOS model.
hbos = COPOD(contamination=0.01)
hbos.fit(X_train)

# Predict on test data.
y_pred = hbos.predict(X_test)

# Evaluate performance.
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[488   0]
 [  2   5]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       488
           1       1.00      0.71      0.83         7

    accuracy                           1.00       495
   macro avg       1.00      0.86      0.92       495
weighted avg       1.00      1.00      1.00       495



In [5]:
from pyod.models.hbos import HBOS
from pyod.models.copod import COPOD
from pyod.models.iforest import IForest
from pyod.models.ecod import ECOD
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, average_precision_score)
import pandas as pd

import numpy as np
import random

np.random.seed(42)
random.seed(42)


# --- Data Preparation ---
RD = ['Saposin', 'Gaucher', 'ASMD']

features = [
    'ORGANOMEGALY', 'HEPATOSPLENOMEGALY', 'HEPATOMEGALY', 'SPLENOMEGALY',
    'WBC', 'Hgb', 'PLATELET', 'NEUTRO_AUTO',
    'GROWTH_RETARDATION', 'FAILURE_TO_THRIVE',
    'INTERSTITIAL_LUNG_DISEASE', 'HYPOTONIA', 'DEVELOPMENTAL_DELAY'
]

disease_choice = 'Saposin'  # Options: 'Saposin', 'Gaucher', 'ASMD', or 'all'

dfs = []
for rd in RD:
    df = pd.read_excel(f'{rd}-results/{rd}_results_v3.xlsx')
    df['Disease'] = rd
    dfs.append(df)
combined_df = pd.concat(dfs, ignore_index=True)
if disease_choice != 'all':
    combined_df = combined_df[combined_df['Disease'] == disease_choice]

# --- Remove Duplicate Controls ---
# Assume the unique identifier for patients is 'PERSONID'.
# Controls (LABEL == 0) are expected to be the same across datasets.
combined_controls = combined_df[combined_df['LABEL'] == 0].drop_duplicates(subset='PERSONID')
combined_positives = combined_df[combined_df['LABEL'] == 1]
combined_df = pd.concat([combined_controls, combined_positives], ignore_index=True)

# Separate negatives (controls) and positives.
neg_df = combined_df[combined_df['LABEL'] == 0].copy()
pos_df = combined_df[combined_df['LABEL'] == 1].copy()

# Set contamination as the proportion of positives in the combined data.
CONTAMINATION = pos_df.shape[0] / combined_df.shape[0]
#CONTAMINATION = 0.1

# Reserve a fraction of negatives for testing.
test_neg_fraction = 0.05
neg_test = neg_df.sample(frac=test_neg_fraction, random_state=42)
neg_train = neg_df.drop(neg_test.index)
print(f"Negatives: Train={neg_train.shape[0]}, Test={neg_test.shape[0]}")

# Build test set: all positives plus sampled negatives.
test_df = pd.concat([pos_df, neg_test], ignore_index=True)
train_df = neg_train  # Training: only negatives.

X_train = train_df[features].copy()
X_test = test_df[features].copy()

# --- Imputation for Training Set (Negatives Only) ---
X_train_imputed = X_train.copy()
num_cols = X_train_imputed.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train_imputed.select_dtypes(exclude=['number']).columns.tolist()

for col in num_cols:
    impute_val = X_train_imputed[col].mean()
    X_train_imputed[col] = X_train_imputed[col].fillna(impute_val)

for col in cat_cols:
    impute_val = X_train_imputed[col].mode()[0]
    X_train_imputed[col] = X_train_imputed[col].fillna(impute_val)

# --- Imputation for Test Set (Positives and Negatives Separately) ---
y_test = test_df['LABEL'].copy()  # Define test labels before imputation.
X_test_imputed = X_test.copy()
mask_neg = (y_test == 0)
mask_pos = (y_test == 1)

for col in num_cols:
    neg_impute = X_train_imputed[col].mean()  # Use training negatives mean.
    X_test_imputed.loc[mask_neg, col] = X_test_imputed.loc[mask_neg, col].fillna(neg_impute)
    pos_impute = X_test_imputed.loc[mask_pos, col].mean()  # Compute mean for positives only.
    X_test_imputed.loc[mask_pos, col] = X_test_imputed.loc[mask_pos, col].fillna(pos_impute)

for col in cat_cols:
    neg_mode = X_train_imputed[col].mode()[0]
    X_test_imputed.loc[mask_neg, col] = X_test_imputed.loc[mask_neg, col].fillna(neg_mode)
    pos_mode = X_test_imputed.loc[mask_pos, col].mode()[0]
    X_test_imputed.loc[mask_pos, col] = X_test_imputed.loc[mask_pos, col].fillna(pos_mode)

X_train = X_train_imputed.copy()
X_test = X_test_imputed.copy()

# Convert to NumPy arrays for IForest to avoid feature-name warnings.
X_train_np = X_train.to_numpy()
X_test_np = X_test.to_numpy()

# --- Evaluate Multiple Models ---
models = {
    'HBOS': HBOS(contamination=CONTAMINATION),
    'COPOD': COPOD(contamination=CONTAMINATION),
    'IForest': IForest(contamination=CONTAMINATION),
    'ECOD': ECOD(contamination=CONTAMINATION)
}

results = []
for model_name, model in models.items():
    if model_name == 'IForest':
        model.fit(X_train_np)
        y_pred = model.predict(X_test_np)
        scores = model.decision_function(X_test_np)
    else:
        model.fit(X_train)
        y_pred = model.predict(X_test)
        scores = model.decision_function(X_test)
    
    cm = confusion_matrix(y_test, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = cm[0, 0] if cm.shape[0] > 0 and cm.shape[1] > 0 else 0
        fp = cm[0, 1] if cm.shape[1] > 1 else 0
        fn = cm[1, 0] if cm.shape[0] > 1 else 0
        tp = cm[1, 1] if cm.shape[0] > 1 and cm.shape[1] > 1 else 0
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    try:
        roc = roc_auc_score(y_test, scores)
    except ValueError:
        roc = None
    try:
        avg_prec = average_precision_score(y_test, scores)
    except ValueError:
        avg_prec = None
    
    results.append({
        'MODEL NAME': model_name,
        'TP': tp,
        'TN': tn,
        'FN': fn,
        'FP': fp,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC AUC': roc,
        'Average Precision': avg_prec
    })

results_df = pd.DataFrame(results)
print(results_df)


Negatives: Train=514, Test=27
  MODEL NAME  TP  TN  FN  FP  Accuracy  Precision    Recall  F1-Score  \
0       HBOS   2  27   1   0  0.966667        1.0  0.666667       0.8   
1      COPOD   1  27   2   0  0.933333        1.0  0.333333       0.5   
2    IForest   1  27   2   0  0.933333        1.0  0.333333       0.5   
3       ECOD   0  27   3   0  0.900000        0.0  0.000000       0.0   

    ROC AUC  Average Precision  
0  0.987654           0.916667  
1  0.975309           0.866667  
2  0.987654           0.916667  
3  0.987654           0.916667  


In [6]:
results_df.to_excel('Saposin_model_comparison_results.xlsx', index=False)